criando uma tabela de orders com: quantidade de itens, vendedores, mapeamento de status e valor total da venda

In [0]:
select * from 1bronze_bike.orders

In [0]:
select * from 1bronze_bike.staffs

In [0]:
select 
o.staff_id,
s.first_name,
s.last_name
from 1bronze_bike.orders as o
left join 1bronze_bike.staffs as s
on o.staff_id = s.staff_id



In [0]:
SELECT DISTINCT order_status FROM 1bronze_bike.orders

In [0]:
select 
order_id,
order_status,
CASE
    WHEN order_status = 1 then 'Pending'
    WHEN order_status = 2 then 'Processing'
    WHEN order_status = 3 then 'Shipped'
    WHEN order_status = 4 then 'Delivered'
    END AS Status
from 1bronze_bike.orders

In [0]:
select * from 1bronze_bike.order_items limit 10

In [0]:
select * from `2silver_bike`.product 
where product_id=20

In [0]:
select 
order_id,
list_price,
quantity,
discount,
round((list_price*quantity)-(list_price*discount),2) as total_venda --(list_price*quantity) * (1- discount)
from 1bronze_bike.order_items limit 10


In [0]:
with total_venda as (
    select 
       order_id,
       quantity,
       round((list_price*quantity)*(1- discount),2) as total_venda--(list_price*quantity)-(list_price*discount)
       from 1bronze_bike.order_items
    )
SELECT 
o.order_id,
o.customer_id,
o.order_date,
o.required_date,
o.shipped_date,
sto.store_name,
sto.state as store_state,
sto.city as store_city,
sta.first_name as staff_first_name,
sta.email as staff_email,
sta.active as staff_active,
tv.total_venda,
tv.quantity,
CASE
    WHEN o.order_status = 1 then 'Pending'
    WHEN o.order_status = 2 then 'Processing'
    WHEN o.order_status =3 then 'Shipped'
    WHEN o.order_status = 4 then 'Delivered'
    else 'Unknown'
    END AS Status
from 1bronze_bike.orders  as o

left join 1bronze_bike.stores as sto
on o.store_id = sto.store_id

left join 1bronze_bike.staffs as sta
on o.staff_id = sta.staff_id

left join total_venda as tv
on o.order_id = tv.order_id

                 

In [0]:
%python

df_orders_silver = spark.sql(
    """
with total_venda as (
    select 
       order_id,
       quantity,
       round((list_price*quantity)*(1- discount),2) as total_venda--(list_price*quantity)-(list_price*discount)
       from 1bronze_bike.order_items
    )
SELECT 
o.order_id,
o.customer_id,
o.order_date,
o.required_date,
sto.store_name,
sto.state as store_state,
sto.city as store_city,
sta.first_name as staff_first_name,
sta.email as staff_email,
sta.active as staff_active,
tv.total_venda,
tv.quantity,
CASE
    WHEN o.order_status = 1 then 'Pending'
    WHEN o.order_status = 2 then 'Processing'
    WHEN o.order_status =3 then 'Shipped'
    WHEN o.order_status = 4 then 'Delivered'
    else 'Unknown'
    END AS Status
from 1bronze_bike.orders  as o

left join 1bronze_bike.stores as sto
on o.store_id = sto.store_id

left join 1bronze_bike.staffs as sta
on o.staff_id = sta.staff_id

left join total_venda as tv
on o.order_id = tv.order_id

                    """
)

display(df_orders_silver)

In [0]:
%python
df_orders_silver.write\
.mode('overwrite')\
.format('delta')\
.option('mergeSchema','true')\
.saveAsTable('2silver_bike.orders')

In [0]:
select * from 2silver_bike.orders limit 10